# BiLSTM baseline (MultiRC)
Structure: For each (paragraph, question, candidate answer), the model predicts whether the answer is correct.

We're using **BiLSTM**, **BiGRU** and **Transformer**. To compare them fairly, **only the encoder changes**. Tokenisation, the train/validation split, hyperparameters, the training loop and the evaluation are identical.

**To make the other versions:**
1. Copy this notebook.
2. Change `MODEL_NAME` in section 0.
3. Replace the encoder class in **section 4**.

I think we just leave everything else the same? 

Model selection (the best epoch) uses the **validation** split only. The test set is scored once, at the end.

## 0. Settings (shared: keep identical across all three models)

In [1]:
import json, re, random, time
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

DATA_DIR = "./MultiRC/"   # Colab/Drive: change to the folder that holds the two MultiRC json files
MODEL_NAME = "bilstm"    

SEED = 42
VAL_SIZE = 0.1            # share of TRAINING paragraphs held out for validation (same split as the DistilBERT baseline)
MIN_FREQ = 2              # words seen fewer times in training become <unk>
MAX_PASSAGE_LEN = 640     # tokens; covers every paragraph, so nothing is cut
MAX_QA_LEN = 112          # tokens for "question <sep> answer"; covers every example
EMB_DIM = 200
HIDDEN_DIM = 128          # per direction; every encoder must output 2 * HIDDEN_DIM features per token
NUM_LAYERS = 1
DROPOUT = 0.3
BATCH_SIZE = 32
LR = 1e-3
WEIGHT_DECAY = 0.01
EPOCHS = 15
PATIENCE = 3              # stop early after this many epochs without validation improvement


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")

device: NVIDIA GeForce GTX 1660 Ti


## 1. Load data (shared)
One row per answer option. The validation set is 10% of the *training* paragraphs, split by paragraph so the same passage never appears in both.

In [2]:
def clean_text(text):
    return re.sub(r"\s+", " ", text).strip()


def clean_paragraph(html):
    return clean_text(re.sub(r"<b>Sent \d+: </b>", " ", html).replace("<br>", " "))


def load_split(path):
    with open(path, encoding="utf-8") as f:
        data = json.load(f)["data"]
    rows = []
    for item in data:
        paragraph = clean_paragraph(item["paragraph"]["text"])
        for q_idx, question in enumerate(item["paragraph"]["questions"]):
            for answer in question["answers"]:
                rows.append({"para_id": item["id"], "q_id": f"{item['id']}_{q_idx}", "paragraph": paragraph,
                             "question": clean_text(question["question"]), "answer": clean_text(answer["text"]),
                             "label": int(answer["isAnswer"])})
    return pd.DataFrame(rows)


train_full = load_split(DATA_DIR + "train_456-fixedIds.json")
test_df = load_split(DATA_DIR + "test_83-fixedIds.json")
train_paras, val_paras = train_test_split(train_full["para_id"].unique(), test_size=VAL_SIZE, random_state=SEED)
train_df = train_full[train_full["para_id"].isin(train_paras)].reset_index(drop=True)
val_df = train_full[train_full["para_id"].isin(val_paras)].reset_index(drop=True)
for name, df in (("train", train_df), ("val", val_df), ("test", test_df)):
    print(f"{name:5s}: {df.para_id.nunique():3d} paragraphs, {df.q_id.nunique():4d} questions, "
          f"{len(df):5d} answer options, {100 * df.label.mean():.1f}% correct")

train: 410 paragraphs, 4636 questions, 24634 answer options, 44.2% correct
val  :  46 paragraphs,  495 questions,  2609 answer options, 43.9% correct
test :  83 paragraphs,  953 questions,  4848 answer options, 42.8% correct


## 2. Tokenisation and vocabulary (Shared)
Lower-cased word tokens. The vocabulary comes from the training split only; embeddings are learned from scratch.

In [3]:
TOKEN_RE = re.compile(r"[a-z0-9]+(?:'[a-z]+)?|[^\sa-z0-9]")
PAD, UNK, SEP = 0, 1, 2


def tokenize(text):
    return TOKEN_RE.findall(text.lower())


def build_vocab(df, min_freq=MIN_FREQ):
    counts = Counter()
    for text in pd.concat([df.drop_duplicates("para_id").paragraph, df.drop_duplicates("q_id").question, df.answer]):
        counts.update(tokenize(text))
    words = ["<pad>", "<unk>", "<sep>"] + sorted(w for w, c in counts.items() if c >= min_freq)
    return {w: i for i, w in enumerate(words)}


def encode(text, max_len):
    return [vocab.get(w, UNK) for w in tokenize(text)][:max_len]


vocab = build_vocab(train_df)
print(f"vocabulary: {len(vocab)} words")
print(tokenize(train_df.question[0])[:10], "->", encode(train_df.question[0], 10))

vocabulary: 11707 words
['does', 'the', 'author', 'claim', 'the', 'animated', 'films', 'message', 'is', 'that'] -> [3341, 10526, 1156, 2167, 10526, 834, 4215, 6734, 5645, 10520]


## 3. Dataset and batching (Shared)
Each example is two token sequences: the **passage**, and **"question `<sep>` answer"**, plus the 0/1 label.

In [4]:
class MultiRCDataset(Dataset):
    def __init__(self, df):
        passage_cache = {}
        self.passages, self.qas = [], []
        for row in df.itertuples(index=False):
            if row.para_id not in passage_cache:
                passage_cache[row.para_id] = encode(row.paragraph, MAX_PASSAGE_LEN)
            self.passages.append(passage_cache[row.para_id])
            self.qas.append((encode(row.question, MAX_QA_LEN) + [SEP] + encode(row.answer, MAX_QA_LEN))[:MAX_QA_LEN])
        self.labels = df["label"].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return self.passages[i], self.qas[i], self.labels[i]


def pad_batch(seqs):
    out = torch.full((len(seqs), max(1, max(len(s) for s in seqs))), PAD, dtype=torch.long)
    for i, s in enumerate(seqs):
        out[i, :len(s)] = torch.tensor(s, dtype=torch.long)
    return out


def collate(batch):
    passages, qas, labels = zip(*batch)
    return pad_batch(passages), pad_batch(qas), torch.tensor(labels, dtype=torch.float)


train_ds, val_ds, test_ds = MultiRCDataset(train_df), MultiRCDataset(val_df), MultiRCDataset(test_df)

## 4. Encoder (WE CHANGING STUFF HERE)
Every encoder follows the same interface:
* **input:** embedded tokens `x` of shape `[batch, length, EMB_DIM]`, plus `mask` of shape `[batch, length]` (True = real token, False = padding);
* **output:** one vector per token, shape `[batch, length, 2 * HIDDEN_DIM]`, with `self.out_dim = 2 * HIDDEN_DIM`.

For the BiGRU, swap `nn.LSTM` for `nn.GRU`. For the Transformer, use `nn.TransformerEncoder` with `d_model = 2 * HIDDEN_DIM`, project `EMB_DIM` up to that size, add position embeddings for up to `MAX_PASSAGE_LEN` positions, and pass `src_key_padding_mask=~mask`.

In [ ]:
class BiLSTMEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.LSTM(EMB_DIM, HIDDEN_DIM, num_layers=NUM_LAYERS, batch_first=True, bidirectional=True,
                           dropout=DROPOUT if NUM_LAYERS > 1 else 0.0)
        self.out_dim = 2 * HIDDEN_DIM

    def forward(self, x, mask):
        lengths = mask.sum(1).clamp(min=1).cpu()  # the LSTM skips the padding
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        out, _ = self.rnn(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        return out  # [batch, length, 2 * HIDDEN_DIM]


def make_encoder():
    return BiLSTMEncoder()

## 5. Classifier around the encoder (shared)
1. The **same encoder** reads the passage and the "question `<sep>` answer" text.
2. Each sequence is squashed into one vector (mean + max over tokens).
3. The two vectors are compared ($[p;\ h;\ p \odot h;\ |p - h|]$).
4. A small MLP outputs one score: is this answer correct?

There is no attention yet. Every passage word counts equally, which we can improve in the future versions. 

In [ ]:
class PairClassifier(nn.Module):
    def __init__(self, encoder, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMB_DIM, padding_idx=PAD)
        self.encoder = encoder
        self.dropout = nn.Dropout(DROPOUT)
        pooled = 2 * encoder.out_dim  # mean + max
        self.classifier = nn.Sequential(nn.Linear(4 * pooled, HIDDEN_DIM), nn.ReLU(),
                                        nn.Dropout(DROPOUT), nn.Linear(HIDDEN_DIM, 1))

    def encode(self, ids):
        mask = ids != PAD
        states = self.encoder(self.dropout(self.embedding(ids)), mask)
        m = mask.unsqueeze(-1)
        mean = (states * m).sum(1) / m.sum(1).clamp(min=1)
        maxed = states.masked_fill(~m, -1e4).max(1).values
        return torch.cat([mean, maxed], -1)

    def forward(self, passage, qa):
        p, h = self.encode(passage), self.encode(qa)
        return self.classifier(torch.cat([p, h, p * h, (p - h).abs()], -1)).squeeze(-1)  # one logit per example

## 6. Training and evaluation (shared)
Metrics (official MultiRC / SuperGLUE):
* **F1a**: F1 over all answer options.
* **EM0**: % of questions where *every* option is labelled correctly.
* **Acc**: per-option accuracy.

An answer counts as predicted correct when its probability is ≥ 0.5. The best epoch is the one with the highest validation **(F1a + EM0) / 2**, which is SuperGLUE's MultiRC score.

Watch two columns while it trains:
* **pred True**: if it sits at ~100% (or ~0%), the model is just guessing one answer. For reference, predicting True for everything scores F1a ≈ 60 / EM0 ≈ 1.
* **AUC**: how well the model ranks correct answers above wrong ones, ignoring the 0.5 cutoff. 50 means random. If AUC is rising but pred True is stuck at 0% or 100%, the model is learning and only the cutoff is off.

In [ ]:
def multirc_scores(df, preds):
    y, p = df["label"].to_numpy(), np.asarray(preds).astype(int)
    tp, fp, fn = int(((y == 1) & (p == 1)).sum()), int(((y == 0) & (p == 1)).sum()), int(((y == 1) & (p == 0)).sum())
    f1a = 2 * tp / (2 * tp + fp + fn) if tp else 0.0
    em0 = pd.Series(y == p).groupby(df["q_id"].to_numpy()).all().mean()
    return {"F1a": 100 * f1a, "EM0": 100 * em0, "Acc": 100 * (y == p).mean(), "pred True %": 100 * p.mean()}


@torch.no_grad()
def predict_probs(model, ds):
    model.eval()
    probs = [torch.sigmoid(model(passage.to(device), qa.to(device))).cpu()
             for passage, qa, _ in DataLoader(ds, batch_size=128, collate_fn=collate)]
    return torch.cat(probs).numpy()


def train_model(model):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss()
    loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
    best_score, best_state, waited, history = -1.0, None, 0, []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        start, total_loss = time.time(), 0.0
        for passage, qa, labels in loader:
            loss = loss_fn(model(passage.to(device), qa.to(device)), labels.to(device))
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total_loss += loss.item() * len(labels)
        val_probs = predict_probs(model, val_ds)
        val = multirc_scores(val_df, val_probs >= 0.5)
        val["AUC"] = 100 * roc_auc_score(val_df["label"], val_probs)
        score = (val["F1a"] + val["EM0"]) / 2
        history.append({"epoch": epoch, "train_loss": total_loss / len(train_ds), **val, "val score": score})
        print(f"epoch {epoch:2d} | train loss {total_loss / len(train_ds):.4f} | val F1a {val['F1a']:.1f}  "
              f"EM0 {val['EM0']:.1f}  Acc {val['Acc']:.1f}  AUC {val['AUC']:.1f} | "
              f"pred True {val['pred True %']:.0f}% | {time.time() - start:.0f}s")
        if score > best_score:
            best_score, waited = score, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            waited += 1
            if waited >= PATIENCE:
                print(f"no improvement for {PATIENCE} epochs: stopping")
                break
    model.load_state_dict(best_state)
    best = max(history, key=lambda h: h["val score"])
    print(f"best epoch {best['epoch']}: val F1a {best['F1a']:.1f}, EM0 {best['EM0']:.1f}")
    return pd.DataFrame(history)

## 7. Train

In [8]:
set_seed()
model = PairClassifier(make_encoder(), len(vocab)).to(device)
print(f"{MODEL_NAME}: {sum(p.numel() for p in model.parameters()):,} parameters")
history = train_model(model)

bilstm: 2,941,721 parameters
epoch  1 | train loss 0.6666 | val F1a 40.3  EM0 6.1  Acc 62.3  AUC 63.3 | pred True 19% | 97s
epoch  2 | train loss 0.6347 | val F1a 45.4  EM0 9.3  Acc 62.6  AUC 60.9 | pred True 24% | 92s
epoch  3 | train loss 0.6067 | val F1a 38.7  EM0 6.1  Acc 62.6  AUC 62.6 | pred True 17% | 94s
epoch  4 | train loss 0.5700 | val F1a 40.8  EM0 6.7  Acc 59.9  AUC 60.6 | pred True 24% | 92s
epoch  5 | train loss 0.5353 | val F1a 49.4  EM0 10.1  Acc 60.5  AUC 61.3 | pred True 34% | 94s
epoch  6 | train loss 0.5083 | val F1a 46.8  EM0 8.9  Acc 59.9  AUC 60.7 | pred True 31% | 88s
epoch  7 | train loss 0.4841 | val F1a 40.4  EM0 7.1  Acc 60.7  AUC 60.7 | pred True 22% | 78s
epoch  8 | train loss 0.4552 | val F1a 44.5  EM0 8.1  Acc 58.9  AUC 58.7 | pred True 30% | 78s
no improvement for 3 epochs: stopping
best epoch 5: val F1a 49.4, EM0 10.1


## 8. Test results (run after training)
This saves the per-answer test predictions and the training history to `results/` so the three models can be compared.

In [9]:
test_probs = predict_probs(model, test_ds)
test = multirc_scores(test_df, test_probs >= 0.5)
print(f"{MODEL_NAME} TEST: F1a {test['F1a']:.1f} | EM0 {test['EM0']:.1f} | Acc {test['Acc']:.1f}")
reference = multirc_scores(test_df, np.ones(len(test_df)))
print(f"(reference, predict all True: F1a {reference['F1a']:.1f} | EM0 {reference['EM0']:.1f} | Acc {reference['Acc']:.1f})")

Path("results").mkdir(exist_ok=True)
test_df.assign(prob=test_probs.round(4), prediction=(test_probs >= 0.5).astype(int))[
    ["para_id", "q_id", "label", "prob", "prediction"]].to_csv(f"results/{MODEL_NAME}_test_preds.csv", index=False)
history.to_csv(f"results/{MODEL_NAME}_history.csv", index=False)

bilstm TEST: F1a 47.6 | EM0 6.4 | Acc 60.2
(reference, predict all True: F1a 59.9 | EM0 0.8 | Acc 42.8)
